# Simple backtest

Walk-forward OLS → residual z-score → enter when `|z| >= ENTRY_Z` (mean-reversion, 1-bar hold).

**Data:** `SOURCE = "live"` (SOFR panel) or `"csv"` (your file / `run_pull_data` export).

Only action knob: **`ENTRY_Z`** (e.g. `1.0` = 1 std).

In [ ]:
import importlib, logging, os, sys
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "backtest_lab.py").exists() else _DZ
if not (REPO / "backtest_lab.py").exists():
    raise FileNotFoundError("No backtest_lab.py — git pull origin main")
os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz
importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

## Inputs

In [ ]:
# Data
SOURCE = "live"   # "live" | "csv"
CSV_PATH = "exports/panels/my_panel.csv"

# Model
TARGET = "5s7s10s"
FEATURES = ["10s30s"]   # e.g. ["CL1", "USGG5YR"] if those cols exist
TRAIN_SIZE = 50

# Strategy — only entry threshold
ENTRY_Z = 1.0           # enter when |z_resid| >= ENTRY_Z
Z_LOOKBACK = 50

START = "2024-01-01"
END = None

In [ ]:
import backtest_lab
importlib.reload(backtest_lab)

panel = backtest_lab.load_panel_for_backtest(SOURCE, CSV_PATH)
print("panel", panel.shape, "cols sample:", list(panel.columns)[:15])

path, summary = backtest_lab.run_simple_backtest(
    panel,
    target=TARGET,
    features=FEATURES,
    entry_z=ENTRY_Z,
    z_lookback=Z_LOOKBACK,
    train_size=TRAIN_SIZE,
    start=START,
    end=END,
)
files = backtest_lab.save_lab_outputs(path, summary)
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
display(path.tail(8))

In [ ]:
pngs = [p for p in files if str(p).endswith(".png")]
for p in pngs:
    display(Image(filename=str(p)))